In [ ]:
#question 2
import json, subprocess
import pandas as pd
pd.set_option("display.max_colwidth", None)
LABELS = ["BACKWARD_ENTAILMENT", "EQUIVALENCE", "FORWARD_ENTAILMENT", "NEGATIVE_OTHER"]
GOLD = "final_data/dev/dico_nli_dev_track1_reference.csv"
ref = pd.read_csv(GOLD)

def confusion(pred_path):
    r = subprocess.run(["python3", "-m", "evaluation_functions", "--gold", GOLD,
                        "--predictions", pred_path, "--output-dir", "results/tmp_scores"], capture_output=True, text=True)
    cm = json.loads(r.stdout)["classification"]["confusion_matrix"]
    return pd.DataFrame(cm).T.loc[LABELS, LABELS]

for name in ["deberta_seed13", "qwen_zeroshot", "qwen_fewshot"]:
    print(name, "(rows = gold, columns = predicted)")
    print(confusion(f"results/{name}_dev.csv"), "\n")

deberta_seed13 (rows = gold, columns = predicted)
                     BACKWARD_ENTAILMENT  EQUIVALENCE  FORWARD_ENTAILMENT  \
BACKWARD_ENTAILMENT                  165           11                   2   
EQUIVALENCE                            2          142                   6   
FORWARD_ENTAILMENT                     2            8                 169   
NEGATIVE_OTHER                        11           12                   9   

                     NEGATIVE_OTHER  
BACKWARD_ENTAILMENT              12  
EQUIVALENCE                      24  
FORWARD_ENTAILMENT               11  
NEGATIVE_OTHER                   74   

qwen_zeroshot (rows = gold, columns = predicted)
                     BACKWARD_ENTAILMENT  EQUIVALENCE  FORWARD_ENTAILMENT  \
BACKWARD_ENTAILMENT                    2           11                 166   
EQUIVALENCE                            1           73                  95   
FORWARD_ENTAILMENT                     6           66                 112   
NEGATIVE_OTHER 

In [2]:
#question 3
def load(pred_path):
    pred = pd.read_csv(pred_path).rename(columns={"label": "pred"})
    return ref.merge(pred, on="instance_id")

d = load("results/deberta_seed13_dev.csv")
errors = d[d["label"] != d["pred"]]
print("Errors:", len(errors), "of", len(d))
print(errors.groupby(["label", "pred"]).size().sort_values(ascending=False))
errors[["text1", "text2", "label", "pred"]].sample(30, random_state=0)

Errors: 110 of 660
label                pred               
EQUIVALENCE          NEGATIVE_OTHER         24
BACKWARD_ENTAILMENT  NEGATIVE_OTHER         12
NEGATIVE_OTHER       EQUIVALENCE            12
BACKWARD_ENTAILMENT  EQUIVALENCE            11
FORWARD_ENTAILMENT   NEGATIVE_OTHER         11
NEGATIVE_OTHER       BACKWARD_ENTAILMENT    11
                     FORWARD_ENTAILMENT      9
FORWARD_ENTAILMENT   EQUIVALENCE             8
EQUIVALENCE          FORWARD_ENTAILMENT      6
BACKWARD_ENTAILMENT  FORWARD_ENTAILMENT      2
EQUIVALENCE          BACKWARD_ENTAILMENT     2
FORWARD_ENTAILMENT   BACKWARD_ENTAILMENT     2
dtype: int64


,text1,text2,label,pred
581,over child,over daughter,NEGATIVE_OTHER,BACKWARD_ENTAILMENT
70,in a grassy green field,in a yard,FORWARD_ENTAILMENT,NEGATIVE_OTHER
546,game,a video game,BACKWARD_ENTAILMENT,EQUIVALENCE
26,for costa concordia shipwreck disaster,over Costa Concordia disaster,FORWARD_ENTAILMENT,EQUIVALENCE
154,"inside a cow-schemed , fur-covered dome",inside and black and white furry enclosure,FORWARD_ENTAILMENT,NEGATIVE_OTHER
618,with network-sharing deal,"network-sharing , spectrum deal",NEGATIVE_OTHER,BACKWARD_ENTAILMENT
654,A close-up view,Close up,NEGATIVE_OTHER,FORWARD_ENTAILMENT
62,The brownish and white dogs,The tan and white dog,FORWARD_ENTAILMENT,EQUIVALENCE
118,to deliver statement,to speak,FORWARD_ENTAILMENT,EQUIVALENCE
598,onto another,to another,NEGATIVE_OTHER,EQUIVALENCE


In [4]:
#question 4
REV = {"FORWARD_ENTAILMENT": "BACKWARD_ENTAILMENT",
       "BACKWARD_ENTAILMENT": "FORWARD_ENTAILMENT",
       "EQUIVALENCE": "EQUIVALENCE"}
pairs["consistent"] = pairs["pred"] == pairs["pred_flipped"].map(REV)
pairs["both_negative"] = (pairs["pred"] == "NEGATIVE_OTHER") & (pairs["pred_flipped"] == "NEGATIVE_OTHER")

print("SoftCons:", round(pairs["consistent"].mean(), 4))
print("Inconsistent:", (~pairs["consistent"]).sum(), "| both NEGATIVE_OTHER:", pairs["both_negative"].sum())
pairs[pairs["both_negative"]][["text1", "text2", "label"]].head(10)

SoftCons: 0.8664
Inconsistent: 37 | both NEGATIVE_OTHER: 20


,text1,text2,label
16,as his security adviser,as national security adviser,BACKWARD_ENTAILMENT
35,in a yard,in a grassy green field,BACKWARD_ENTAILMENT
65,with a big necklace,with a black top and a necklace,BACKWARD_ENTAILMENT
69,A brownish and white dog,A small orange and white dog,BACKWARD_ENTAILMENT
76,from the airport,of some runway,BACKWARD_ENTAILMENT
77,inside and black and white furry enclosure,"inside a cow-schemed , fur-covered dome",BACKWARD_ENTAILMENT
91,of a television screen,The wall-mounted flat-screen TV,BACKWARD_ENTAILMENT
96,over Bhutto murder,In Bhutto Death,EQUIVALENCE
98,to pay off,of bribing,EQUIVALENCE
107,across the grassy area,through grass,EQUIVALENCE
